# 🛡️ DiaGuard — Phase 1 & 2: Data Exploration, Training & Model Export

Welcome to the training notebook for **DiaGuard**! 
Here, we will:
1. Download the official CDC BRFSS Diabetes dataset (253,680 records).
2. Explore and clean the data.
3. Train and compare machine learning models (Logistic Regression vs. XGBoost).
4. Generate SHAP explanations (why the model makes each prediction).
5. Export the trained model so you can download it to your laptop for your Web App and API!

### Step 1: Install Required Libraries
We need `ucimlrepo` to fetch the CDC dataset automatically, `xgboost` for our modern classifier, and `shap` to explain predictions.

In [ ]:
!pip install ucimlrepo xgboost shap scikit-learn pandas numpy joblib

### Step 2: Download the CDC Dataset directly into Colab
We use UCI Machine Learning Repository ID `891` (CDC Diabetes Health Indicators). This downloads in seconds directly into Colab memory without filling up your laptop!

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# Fetch the CDC Diabetes Health Indicators dataset (ID 891)
print("⏳ Fetching CDC dataset from UCI repository...")
dataset = fetch_ucirepo(id=891)

# Extract features (X) and target (y)
X = dataset.data.features
y = dataset.data.targets

print(f"✅ Successfully loaded!")
print(f"Total Patient Records: {X.shape[0]:,}")
print(f"Number of Health/Lifestyle Features: {X.shape[1]}")

### Step 3: Inspect the Features and Target
Let's see what features are in this dataset and check the balance between healthy vs. diabetic individuals.

In [ ]:
# View the first 5 records
display(X.head())

# Check target distribution (0 = No Diabetes, 1 = Pre-diabetic/Diabetic)
target_counts = y.value_counts()
print("\n--- Target Distribution ---")
print(target_counts)
print(f"Percentage with Diabetes or Pre-diabetes: {(target_counts.iloc[1] / len(y)) * 100:.2f}%")

### Step 4: Split into Training and Testing Sets
We use an 80/20 train/test split. We also use `stratify=y` so both sets have the exact same proportion of diabetic cases.

In [ ]:
from sklearn.model_selection import train_test_split

# Ensure y is a 1D array
y_flat = y.values.ravel()

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_flat, test_size=0.2, random_state=42, stratify=y_flat
)

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Testing rows: {X_test.shape[0]:,}")

### Step 5: Train Models (Baseline vs. XGBoost)
We train two models:
1. **Logistic Regression**: A simple linear baseline.
2. **XGBoost Classifier**: An industry-standard gradient boosted tree model that captures non-linear risk factors.

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1. Baseline Model: Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")

# 2. Advanced Model: XGBoost (with scale_pos_weight to handle class imbalance)
print("\nTraining XGBoost Classifier...")
# Scale pos weight helps detect positive diabetic cases accurately
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
imbalance_ratio = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=5,
    scale_pos_weight=imbalance_ratio,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

print(f"\n⭐ XGBoost ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}")
print("\n--- Detailed XGBoost Classification Report ---")
print(classification_report(y_test, xgb_preds, target_names=["Healthy", "Diabetic/Pre-diabetic"]))

### Step 6: Explainable AI with SHAP
This reveals which lifestyle and health habits have the biggest impact on increasing or decreasing diabetes risk.

In [ ]:
import shap
import matplotlib.pyplot as plt

# Sample a small batch (1000 rows) for fast SHAP computation
X_sample = X_test.sample(1000, random_state=42)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)

# Plot the most influential risk factors
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)

### Step 7: Export the Model & Metadata
We bundle the trained XGBoost model along with feature names and default thresholds into `diaguard_model.joblib`.
Running this cell will automatically download the lightweight file (~2 MB) straight to your computer!

In [ ]:
import joblib
from google.colab import files

# Package model and metadata
model_package = {
    "model": xgb_model,
    "feature_names": list(X.columns),
    "threshold": 0.5,
    "description": "DiaGuard XGBoost Classifier trained on CDC BRFSS dataset (253,680 records)"
}

# Save locally in Colab
joblib.dump(model_package, "diaguard_model.joblib")
print("✅ Model saved as diaguard_model.joblib")

# Trigger browser download to your laptop
files.download("diaguard_model.joblib")